##### READ THE FILE BreastTissue.xlsx

In [1]:
import pandas as pd
from sklearn.metrics import confusion_matrix
import numpy as np


file_path = '/home/sarantis/Documents/Decision Theory/Project_ΘΑ_2023-24/BreastTissue.xlsx' #path to file BreastTissue

dataframe = pd.read_excel(file_path, sheet_name = 'Data')
print(dataframe)

     Case # Class           I0     PA500       HFS           DA  \
0         1   car   524.794072  0.187448  0.032114   228.800228   
1         2   car   330.000000  0.226893  0.265290   121.154201   
2         3   car   551.879287  0.232478  0.063530   264.804935   
3         4   car   380.000000  0.240855  0.286234   137.640111   
4         5   car   362.831266  0.200713  0.244346   124.912559   
..      ...   ...          ...       ...       ...          ...   
101     102   adi  2000.000000  0.106989  0.105418   520.222649   
102     103   adi  2600.000000  0.200538  0.208043  1063.441427   
103     104   adi  1600.000000  0.071908 -0.066323   436.943603   
104     105   adi  2300.000000  0.045029  0.136834   185.446044   
105     106   adi  2600.000000  0.069988  0.048869   745.474369   

              Area        A/DA      Max IP          DR            P  
0      6843.598481   29.910803   60.204880  220.737212   556.828334  
1      3163.239472   26.109202   69.717361   99.084964 

##### ΑΝΤΙΣΤΟΙΧΙΣΗ ΚΛΑΣΕΩΝ ΜΕ ΤΙΜΕΣ ΜΕΣΩ REGULAR EXPRESSION

In [2]:
dataframe = dataframe.replace(regex=['car'], value='1')
dataframe = dataframe.replace(regex=['fad'], value='2')
dataframe = dataframe.replace(regex=['mas'], value='3')
dataframe = dataframe.replace(regex=['gla'], value='4')
dataframe = dataframe.replace(regex=['con'], value='5')
dataframe = dataframe.replace(regex=['adi'], value='6')

print(dataframe)

     Case # Class           I0     PA500       HFS           DA  \
0         1     1   524.794072  0.187448  0.032114   228.800228   
1         2     1   330.000000  0.226893  0.265290   121.154201   
2         3     1   551.879287  0.232478  0.063530   264.804935   
3         4     1   380.000000  0.240855  0.286234   137.640111   
4         5     1   362.831266  0.200713  0.244346   124.912559   
..      ...   ...          ...       ...       ...          ...   
101     102     6  2000.000000  0.106989  0.105418   520.222649   
102     103     6  2600.000000  0.200538  0.208043  1063.441427   
103     104     6  1600.000000  0.071908 -0.066323   436.943603   
104     105     6  2300.000000  0.045029  0.136834   185.446044   
105     106     6  2600.000000  0.069988  0.048869   745.474369   

              Area        A/DA      Max IP          DR            P  
0      6843.598481   29.910803   60.204880  220.737212   556.828334  
1      3163.239472   26.109202   69.717361   99.084964 

##### ΚΑΝΟΝΙΚΟΠΟΙΗΣΗ ΔΕΔΟΜΕΝΩΝ ΣΤΟ ΕΥΡΟΣ [-1,1]

In [3]:
dataset_columns = dataframe.columns.difference(['Class', 'Case #']) #Βγάζουμε τις στήλες που δεν θέλουμε να κανονικοποιήσουμε

el_times = dataframe[dataset_columns].min()   #Ελάχιστη τιμή
meg_times = dataframe[dataset_columns].max()   #Μέγιστη τιμή

new_min = -1
new_max = 1

dataframe[dataset_columns] = ((dataframe[dataset_columns] - el_times) / (meg_times - el_times)) * (new_max - new_min) + new_min

print(dataframe)

     Case # Class        I0     PA500       HFS        DA      Area      A/DA  \
0         1     1 -0.687212  0.012109 -0.631373 -0.599245 -0.922330 -0.651455   
1         2     1 -0.831665  0.240161  0.241830 -0.805505 -0.964534 -0.698251   
2         3     1 -0.667127  0.272452 -0.513725 -0.530257 -0.864481 -0.467008   
3         4     1 -0.794587  0.320888  0.320261 -0.773916 -0.938860 -0.536512   
4         5     1 -0.807318  0.088799  0.163399 -0.798303 -0.963075 -0.695384   
..      ...   ...       ...       ...       ...       ...       ...       ...   
101     102     6  0.406748 -0.453078 -0.356863 -0.040855 -0.541110 -0.071081   
102     103     6  0.851687  0.087790  0.027451  1.000000  1.000000  1.000000   
103     104     6  0.110122 -0.655903 -1.000000 -0.200425 -0.855686 -0.663118   
104     105     6  0.629218 -0.811302 -0.239216 -0.682316 -0.942482 -0.682025   
105     106     6  0.851687 -0.667003 -0.568627  0.390747 -0.543887 -0.361696   

       Max IP        DR    

##### 5-Fold Cross Validation in SVM

In [4]:
from sklearn.model_selection import KFold, cross_validate, cross_val_predict, cross_val_score
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, f1_score, make_scorer, precision_score, recall_score
from imblearn.metrics import geometric_mean_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Δημιουργία Χ dataframe χωρίς 'Class', 'Case #' και y για την class
X = dataframe.drop(['Class', 'Case #'], axis=1)
y = dataframe['Class']

print("Shape of input data: {} and shape of target variable: {}".format(X.shape, y.shape))

X_train, X_test, Y_train, Y_test = train_test_split(X, y, test_size=0.2, random_state=1)

print("Shape of input data: {}, {} and shape of target variable: {}, {}".format(X_train.shape, X_test.shape, Y_train.shape, Y_test.shape))

kf =KFold(n_splits=5, shuffle=True, random_state=1)

def fivefold_cv(classifier, X, y, kf):
    
    scores = {
        'accuracy': 'accuracy',
        'geometric_mean': make_scorer(geometric_mean_score, average='macro')
    }


    # Model Training, Multiple Scoring Metrics
    cv_scores = cross_validate(estimator=classifier, X=X, y=y, cv=kf, scoring=scores, return_train_score=True, verbose=1)

    # Εκτίμηση τιμών 
    predict = cross_val_predict(classifier, X, y, cv=kf)

    # Υπολογσμός Ακρίβειας
    accuracy = accuracy_score(y, predict)

    # Γεωμετρικός Μέσος
    geometric_mean = geometric_mean_score(y, predict)

    # Fit το μοντέλο με τα training data
    classifier.fit(X_train, Y_train)
    test_pred = classifier.predict(X_test)
    test_accuracy = accuracy_score(Y_test, test_pred)

    print("REPORT CLASSIFICATION FOR train data with repeating 5fold: \n",classification_report(Y_train, predict))

    # Αποτελέσματα
    print("Mean Training accuracy:", cv_scores['train_accuracy'].mean())
    print("Mean Training geometric_mean:", cv_scores['train_geometric_mean'].mean())
    print()
    print("Mean Validation accuracy:", cv_scores['test_accuracy'].mean())
    print("Mean Validation geometric_mean:", cv_scores['test_geometric_mean'].mean())
    print()
    print("Ακρίβεια κατά το cross validation", accuracy)
    print("Geometric Mean: {:.5f}".format(geometric_mean))

Shape of input data: (106, 9) and shape of target variable: (106,)
Shape of input data: (84, 9), (22, 9) and shape of target variable: (84,), (22,)


### Naive Bayes Classifier

In [5]:
# Naive Bayes instance
naive_bayes = GaussianNB()

naive_bayes_score = fivefold_cv(naive_bayes, X_train, Y_train, kf)

REPORT CLASSIFICATION FOR train data with repeating 5fold: 
               precision    recall  f1-score   support

           1       0.81      0.85      0.83        20
           2       0.39      0.58      0.47        12
           3       0.33      0.21      0.26        14
           4       0.30      0.27      0.29        11
           5       0.75      0.82      0.78        11
           6       0.93      0.81      0.87        16

    accuracy                           0.62        84
   macro avg       0.59      0.59      0.58        84
weighted avg       0.62      0.62      0.61        84

Mean Training accuracy: 0.7142230026338894
Mean Training geometric_mean: 0.8173937989695347

Mean Validation accuracy: 0.6183823529411765
Mean Validation geometric_mean: 0.7034895001457019

Ακρίβεια κατά το cross validation 0.6190476190476191
Geometric Mean: 0.51775


### Support Vector Machine Classifier

##### Υπολογισμός Παραμέτρου C

In [6]:
from sklearn import svm
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV

# Διάστημα για το C [1, 200] με βήμα 5
C_space = np.arange(1, 200, 5)

kernels=['rbf']
param_grid = {'C': C_space,'kernel': kernels}

# SVM μοντέλο με Radial Basis Kernel Function
svm = SVC(kernel='rbf')

# Δημιουργία του gridsearch για εύρεση C βάσει μέσης ακρίβειας σε κάθε fold
grid_search = GridSearchCV(svm, param_grid={'C': C_space}, cv=5, scoring='accuracy', refit=True, verbose=1)
grid_search.fit(X_train, Y_train)

best_C = grid_search.best_params_['C']
print("Best C:", best_C)

best_score = grid_search.best_score_
print("Best Score:", best_score)

# Μέσες τιμές για κάθε C value
mean_test_scores = grid_search.cv_results_['mean_test_score']
params = grid_search.cv_results_['params']

for param, score in zip(params, mean_test_scores):
    print(f"C: {param['C']}, Mean Test Score: {score * 100:.2f}%")

Fitting 5 folds for each of 40 candidates, totalling 200 fits
Best C: 46
Best Score: 0.6911764705882353
C: 1, Mean Test Score: 64.26%
C: 6, Mean Test Score: 60.74%
C: 11, Mean Test Score: 64.34%
C: 16, Mean Test Score: 61.99%
C: 21, Mean Test Score: 63.24%
C: 26, Mean Test Score: 64.41%
C: 31, Mean Test Score: 64.41%
C: 36, Mean Test Score: 65.59%
C: 41, Mean Test Score: 67.94%
C: 46, Mean Test Score: 69.12%
C: 51, Mean Test Score: 66.69%
C: 56, Mean Test Score: 66.69%
C: 61, Mean Test Score: 65.44%
C: 66, Mean Test Score: 66.69%
C: 71, Mean Test Score: 64.26%
C: 76, Mean Test Score: 64.26%
C: 81, Mean Test Score: 63.09%
C: 86, Mean Test Score: 63.01%
C: 91, Mean Test Score: 63.01%
C: 96, Mean Test Score: 63.01%
C: 101, Mean Test Score: 63.01%
C: 106, Mean Test Score: 63.01%
C: 111, Mean Test Score: 63.01%
C: 116, Mean Test Score: 61.84%
C: 121, Mean Test Score: 61.84%
C: 126, Mean Test Score: 61.84%
C: 131, Mean Test Score: 61.84%
C: 136, Mean Test Score: 61.84%
C: 141, Mean Test Scor

##### Υπολογισμός Παραμέτρου γ

In [7]:
# Διάστημα για το γ [0, 10] με βήμα 0.5
g = np.arange(0, 10.5, 0.5)

kernels=['rbf']
param_grid = {'C': best_C, 'gamma':g, 'kernel': kernels}

# SVM μοντέλο με Radial Basis Kernel Function
svm = SVC(kernel='rbf')

# Δημιουργία του gridsearch για εύρεση γ βάσει μέσης ακρίβειας σε κάθε fold
svm_grid_search = GridSearchCV(svm, param_grid={'gamma': g}, cv=5, scoring='accuracy', refit=True, verbose=1)
svm_grid_search.fit(X_train, Y_train)

best_gamma = svm_grid_search.best_params_['gamma']
print("Best Gamma:", best_gamma)

best_score = svm_grid_search.best_score_
print("Best Score:", best_score)

# Μέσες τιμές για κάθε γ value
mean_test_scores = svm_grid_search.cv_results_['mean_test_score']
params = svm_grid_search.cv_results_['params']

for param, score in zip(params, mean_test_scores):
    print(f"gamma: {param['gamma']}, Mean Test Score: {score * 100:.2f}%")

Fitting 5 folds for each of 21 candidates, totalling 105 fits
Best Gamma: 5.0
Best Score: 0.7029411764705882
gamma: 0.0, Mean Test Score: 23.82%
gamma: 0.5, Mean Test Score: 63.09%
gamma: 1.0, Mean Test Score: 64.26%
gamma: 1.5, Mean Test Score: 64.26%
gamma: 2.0, Mean Test Score: 63.16%
gamma: 2.5, Mean Test Score: 65.51%
gamma: 3.0, Mean Test Score: 67.94%
gamma: 3.5, Mean Test Score: 67.94%
gamma: 4.0, Mean Test Score: 67.94%
gamma: 4.5, Mean Test Score: 69.12%
gamma: 5.0, Mean Test Score: 70.29%
gamma: 5.5, Mean Test Score: 69.12%
gamma: 6.0, Mean Test Score: 69.12%
gamma: 6.5, Mean Test Score: 67.87%
gamma: 7.0, Mean Test Score: 66.69%
gamma: 7.5, Mean Test Score: 66.69%
gamma: 8.0, Mean Test Score: 66.69%
gamma: 8.5, Mean Test Score: 65.51%
gamma: 9.0, Mean Test Score: 65.51%
gamma: 9.5, Mean Test Score: 65.51%
gamma: 10.0, Mean Test Score: 65.51%


##### Υλοποίηση Μοντέλου SVM

In [8]:
svm_model = SVC(kernel='rbf', C=best_C, gamma=best_gamma)

smv_scores = fivefold_cv(svm_model, X_train, Y_train, kf)

REPORT CLASSIFICATION FOR train data with repeating 5fold: 
               precision    recall  f1-score   support

           1       0.80      0.80      0.80        20
           2       0.50      0.58      0.54        12
           3       0.42      0.36      0.38        14
           4       0.67      0.73      0.70        11
           5       1.00      0.64      0.78        11
           6       0.84      1.00      0.91        16

    accuracy                           0.70        84
   macro avg       0.70      0.68      0.69        84
weighted avg       0.71      0.70      0.70        84

Mean Training accuracy: 1.0
Mean Training geometric_mean: 1.0

Mean Validation accuracy: 0.7
Mean Validation geometric_mean: 0.7995247413763465

Ακρίβεια κατά το cross validation 0.7023809523809523
Geometric Mean: 0.65244


### Ταξινομητής ΚΝΝ

##### Εύρεση Βέλτισης Παραμέτρου Κ

In [9]:
from sklearn.neighbors import KNeighborsClassifier

# Διάστημα για το K [3, 15] 
K_space = np.arange(3, 16)
param_grid = {'n_neighbors': K_space}

# Νέο KNN μοντέλο 
knn = KNeighborsClassifier()

# Δημιουργία του gridsearch για εύρεση C βάσει μέσης ακρίβειας σε κάθε fold
knn_grid_search = GridSearchCV(knn, param_grid, cv=5, scoring='accuracy', verbose=1)
knn_grid_search.fit(X_train, Y_train)

best_K = knn_grid_search.best_params_['n_neighbors']
print("Best K:", best_K)

best_score = knn_grid_search.best_score_
print("Best Score:", best_score)

# Print mean_scores for each gamma value
mean_test_scores = knn_grid_search.cv_results_['mean_test_score']
params = knn_grid_search.cv_results_['params']

for param, score in zip(params, mean_test_scores):
    print(f"n_neighbors: {param['n_neighbors']}, Mean Test Score: {score * 100:.2f}%")

Fitting 5 folds for each of 13 candidates, totalling 65 fits
Best K: 3
Best Score: 0.7279411764705882
n_neighbors: 3, Mean Test Score: 72.79%
n_neighbors: 4, Mean Test Score: 72.57%
n_neighbors: 5, Mean Test Score: 66.54%
n_neighbors: 6, Mean Test Score: 64.19%
n_neighbors: 7, Mean Test Score: 64.26%
n_neighbors: 8, Mean Test Score: 64.26%
n_neighbors: 9, Mean Test Score: 63.09%
n_neighbors: 10, Mean Test Score: 64.26%
n_neighbors: 11, Mean Test Score: 61.99%
n_neighbors: 12, Mean Test Score: 59.63%
n_neighbors: 13, Mean Test Score: 60.81%
n_neighbors: 14, Mean Test Score: 63.09%
n_neighbors: 15, Mean Test Score: 59.41%


##### Υλοποίηση Μοντέλου ΚΝΝ

In [10]:
knn_model = KNeighborsClassifier(n_neighbors=best_K)

knn_scores = fivefold_cv(knn_model, X_train, Y_train, kf)

REPORT CLASSIFICATION FOR train data with repeating 5fold: 
               precision    recall  f1-score   support

           1       0.75      0.90      0.82        20
           2       0.62      0.67      0.64        12
           3       0.38      0.36      0.37        14
           4       0.60      0.55      0.57        11
           5       1.00      0.55      0.71        11
           6       0.89      1.00      0.94        16

    accuracy                           0.70        84
   macro avg       0.71      0.67      0.67        84
weighted avg       0.71      0.70      0.69        84

Mean Training accuracy: 0.8094381035996487
Mean Training geometric_mean: 0.8685832544522984

Mean Validation accuracy: 0.7014705882352941
Mean Validation geometric_mean: 0.7737479258963939

Ακρίβεια κατά το cross validation 0.7023809523809523
Geometric Mean: 0.63205


##### Υλοποίηση Student t-test

In [11]:
from scipy.stats import ttest_ind

attributes = dataframe.columns[(dataframe.columns != 'Class') & (dataframe.columns != 'Case #')]  

carcinoma_df = dataframe[dataframe['Class'] == '1'] 
non_cancer_df = dataframe[dataframe['Class'] != '1'] 

importance = []

for attribute in attributes:
    t_value, p_value = ttest_ind(carcinoma_df[attribute], non_cancer_df[attribute])
    importance.append((attribute, t_value, p_value))

importance.sort(key=lambda x: abs(x[1]), reverse=True)

for attribute, t_value, p_value in importance:
    print(f"{attribute}: t-value = {t_value}, p-value = {p_value}")

#dataframe με τα 4 πιο σημαντικά
top_4 = []
for attribute, _, _ in importance[:4]:
    top_4.append(attribute)
top_4_df = dataframe[['Class'] + top_4]

print(top_4_df)

PA500: t-value = 10.766461705312985, p-value = 1.306591261985479e-18
HFS: t-value = 3.7264605519699012, p-value = 0.00031608064080412834
I0: t-value = -2.727208358962709, p-value = 0.0074985265688774
P: t-value = -2.260715411441866, p-value = 0.025859487123299475
A/DA: t-value = 1.9016625355668326, p-value = 0.0599825769970918
Max IP: t-value = -0.6808080513228802, p-value = 0.49750594517502433
DA: t-value = -0.5961974588862743, p-value = 0.5523387736661007
Area: t-value = -0.4422871085625348, p-value = 0.6591998864063924
DR: t-value = -0.38518253512087836, p-value = 0.7008888319353418
    Class     PA500       HFS        I0         P
0       1  0.012109 -0.631373 -0.687212 -0.688376
1       1  0.240161  0.241830 -0.831665 -0.801381
2       1  0.272452 -0.513725 -0.667127 -0.616258
3       1  0.320888  0.320261 -0.794587 -0.733928
4       1  0.088799  0.163399 -0.807318 -0.783650
..    ...       ...       ...       ...       ...
101     6 -0.453078 -0.356863  0.406748  0.416992
102    

Νέο Dataset


In [12]:
# Δημιουργία Χ dataframe χωρίς 'Class', 'Case #' και y για την class
Xn = top_4_df.drop(['Class'], axis=1)
yn = top_4_df['Class']

print("Shape of input data: {} and shape of target variable: {}".format(Xn.shape, yn.shape))

Xn_train, Xn_test, Yn_train, Yn_test = train_test_split(Xn, yn, test_size=0.2, random_state=1)

Shape of input data: (106, 4) and shape of target variable: (106,)


##### Υλοποίηση Μοντέλου SVM

In [13]:
svm_model_new = SVC(kernel='rbf', C=best_C, gamma=best_gamma)

smv_scores_new = fivefold_cv(svm_model_new, Xn_train, Yn_train, kf)

REPORT CLASSIFICATION FOR train data with repeating 5fold: 
               precision    recall  f1-score   support

           1       0.74      0.70      0.72        20
           2       0.46      0.50      0.48        12
           3       0.15      0.14      0.15        14
           4       0.40      0.55      0.46        11
           5       0.88      0.64      0.74        11
           6       0.94      0.94      0.94        16

    accuracy                           0.60        84
   macro avg       0.59      0.58      0.58        84
weighted avg       0.61      0.60      0.60        84

Mean Training accuracy: 0.9761633011413519
Mean Training geometric_mean: 0.9851584462028222

Mean Validation accuracy: 0.5948529411764707
Mean Validation geometric_mean: 0.7180040855291263

Ακρίβεια κατά το cross validation 0.5952380952380952
Geometric Mean: 0.50339
